# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

from feature_engine.selection import DropFeatures

## Utils

In [2]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
0,0.50,0.203815,0.328482,0.154734,1.585901,-0.308849,1.487008,-1.486487,1.666667,1.040769,...,0.947453,0.005655,0.020178,0.000549,0.997580,0.001872,0.012740,0.40,0,0.781831
1,0.50,0.199575,0.327181,0.176114,0.229628,-1.066602,0.033534,1.440545,-1.000000,-5.009333,...,0.074107,0.081854,0.098684,0.015175,0.007697,0.977128,0.013316,0.20,1,0.885456
2,0.50,0.218840,0.326837,0.187526,2.116616,0.638343,1.902200,-1.486487,1.000000,-1.970285,...,0.226475,0.059038,0.614716,0.024077,0.912645,0.063278,0.014095,0.65,0,0.537300
3,0.25,0.200526,0.100888,0.145166,-1.244581,-0.498287,-1.029455,-0.510810,0.000000,0.338641,...,0.088099,0.041344,0.113803,0.950726,0.010153,0.039121,0.010598,0.35,0,0.239316
4,0.50,0.192082,0.327657,0.214557,0.170660,-1.445478,0.092589,-1.486487,1.000000,0.169816,...,0.256086,0.040890,0.083385,0.009589,0.004844,0.985566,0.009270,0.10,1,0.906308


In [5]:
X_test.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
439140,0.25,0.182308,0.101132,0.133462,-0.124182,-1.066602,0.261317,-0.510810,0.000000,0.396609,...,0.001144,0.000465,0.000894,0.067702,0.028176,0.904122,0.010708,0.20,0,0.954721
439141,0.25,0.356316,0.101132,0.150547,0.052723,-1.634917,0.300590,-0.510810,0.000000,0.470778,...,0.081596,0.033933,0.053686,0.019477,0.011500,0.969023,0.011005,0.05,0,0.963550
439142,0.25,0.121796,0.101132,0.133462,0.052723,0.259466,0.489100,-0.510810,0.000000,0.301036,...,0.087683,0.072629,0.701444,0.038897,0.031448,0.929655,0.010768,0.55,0,0.992709
439143,0.00,0.198847,0.193475,0.253713,-1.008708,1.017219,-1.025511,0.464868,0.333333,0.724449,...,0.023113,0.022859,0.935769,0.984826,0.002887,0.012287,0.010530,0.75,0,0.242362
439144,0.50,0.197127,0.327536,0.114051,1.703838,0.448905,1.518343,0.464868,2.333333,0.003617,...,0.718809,0.040632,0.151701,0.002043,0.991340,0.006618,0.010090,0.60,0,0.766044


# Machine Learning

## Base

In [6]:
cv_results = cross_val_score(
    estimator=RandomForestClassifier(class_weight='balanced', verbose=0), 
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [7]:
cv_results.mean()

np.float64(0.940842688342423)

In [8]:
model = RandomForestClassifier(class_weight='balanced', verbose=0)
model.fit(X_train, y_train.PitNextLap)

RandomForestClassifier(class_weight='balanced')

## Fine Tuning

In [9]:
def objective(trial, X, y):

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        class_weight="balanced",
        random_state=42,
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]
        
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_valid)[:, 1]
        
        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, n_jobs=-1, show_progress_bar=True)


print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-06 10:32:34,395] A new study created in memory with name: no-name-eee708ea-0119-49b4-81a6-86540c50ce06


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-06 11:16:15,764] Trial 9 finished with value: 0.8953649140539508 and parameters: {'n_estimators': 650, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': 'log2', 'bootstrap': True}. Best is trial 9 with value: 0.8953649140539508.
[I 2026-05-06 11:18:53,761] Trial 3 finished with value: 0.9387578226440528 and parameters: {'n_estimators': 229, 'max_depth': 14, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 3 with value: 0.9387578226440528.
[I 2026-05-06 11:32:15,773] Trial 5 finished with value: 0.8791353480779321 and parameters: {'n_estimators': 733, 'max_depth': 4, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False}. Best is trial 3 with value: 0.9387578226440528.
[I 2026-05-06 11:34:55,812] Trial 1 finished with value: 0.9397818295564463 and parameters: {'n_estimators': 384, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_featur

KeyboardInterrupt: 

In [19]:
model_tuned = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    **study.best_params
)

model_tuned.fit(X_train, y_train.PitNextLap)

Pipeline(steps=[('drop_feat',
                 DropFeatures(features_to_drop=['remainder__raceprogress_high',
                                                'remainder__lapnumber_high',
                                                'remainder__position_high',
                                                'remainder__laptime_s_high',
                                                'remainder__position_norm',
                                                'remainder__stint_high',
                                                'remainder__is_pit_lap',
                                                'remainder__lapnumber_low'])),
                ('hist',
                 HistGradientBoostingClassifier(early_stopping=True,
                                                l2_regularization=6.70145048510531,
                                                learning_rate=0.07348770421656471,
                                                max_bins=190, max_depth=10,
                                                max_iter=1000,
                                                min_samples_leaf=81,
                                                n_iter_no_change=50,
                                                random_state=42))])

# Deploy

In [20]:
dump_pickle(pipe_tuned, '../models/model_histgradientboosting.pkl')